In [0]:
gold_path = "/Volumes/ecommerce/default/ecommerce_raw/gold"

In [0]:
#Load silver tables
silver_path = "/Volumes/ecommerce/default/ecommerce_raw/silver"

orders_silver = spark.read.format("delta").load(f"{silver_path}/orders")
order_items_silver = spark.read.format("delta").load(f"{silver_path}/order_items")
order_payments_silver = spark.read.format("delta").load(f"{silver_path}/order_payments")
order_reviews_silver = spark.read.format("delta").load(f"{silver_path}/order_reviews")
customers_silver = spark.read.format("delta").load(f"{silver_path}/customers")
sellers_silver = spark.read.format("delta").load(f"{silver_path}/sellers")
products_silver = spark.read.format("delta").load(f"{silver_path}/products")
category_translation_silver = spark.read.format("delta").load(f"{silver_path}/category_translation")

In [0]:
#Inspect the sales-related tables
orders_silver.printSchema()
order_items_silver.printSchema()
order_payments_silver.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)



In [0]:
#Aggregate order items by order
from pyspark.sql import functions as F

order_items_agg = order_items_silver.groupBy("order_id").agg(
    F.sum("price").alias("total_product_value"),
    F.sum("freight_value").alias("total_freight_value"),
    F.count("order_item_id").alias("total_items")
)

In [0]:
#Aggregate payments by order
order_payments_agg = order_payments_silver.groupBy("order_id").agg(
    F.sum("payment_value").alias("total_payment_value"),
    F.count("payment_sequential").alias("payment_count")
)

In [0]:
#Check Results
print("Orders:", orders_silver.count())
print("Aggregated items:", order_items_agg.count())
print("Aggregated payments:", order_payments_agg.count())

display(order_items_agg.limit(5))
display(order_payments_agg.limit(5))

Orders: 99441
Aggregated items: 98666
Aggregated payments: 99438


order_id,total_product_value,total_freight_value,total_items
00010242fe8c5a6d1ba2dd792cb16214,58.9,13.29,1
00018f77f2f0320c557190d7a144bdd3,239.9,19.93,1
000229ec398224ef6ca0657da4fc703e,199.0,17.87,1
00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1
00042b26cf59d7ce69dfabb4e55b4fd9,199.9,18.14,1


order_id,total_payment_value,payment_count
b81ef226f3fe1789b1e8b2acac839d17,99.33,1
a9810da82917af2d9aefd1278f1dcfa0,24.39,1
25e8ea4e93396b6fa0d3dd708e76c1bd,65.71,1
ba78997921bbcdc1373bb41e913ab953,107.78,1
42fdf880ba16b47b59251dd489d4441a,128.45,1


In [0]:
#Join the three datasets
gold_sales = (
    orders_silver
    .join(order_items_agg, on="order_id", how="left")
    .join(order_payments_agg, on="order_id", how="left")
)

In [0]:
#Inspect the joined data
print("Gold sales count:", gold_sales.count())

gold_sales.printSchema()

display(gold_sales.limit(10))

Gold sales count: 99441
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- total_product_value: double (nullable = true)
 |-- total_freight_value: double (nullable = true)
 |-- total_items: long (nullable = true)
 |-- total_payment_value: double (nullable = true)
 |-- payment_count: long (nullable = true)



order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,total_product_value,total_freight_value,total_items,total_payment_value,payment_count
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,29.99,8.72,1,38.71,3
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,118.7,22.76,1,141.46,1
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,159.9,19.22,1,179.12,1
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,45.0,27.2,1,72.2,1
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z,19.9,8.72,1,28.62,1
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z,147.9,27.36,1,175.26,1
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z,49.9,16.05,1,65.95,1
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z,59.99,15.17,1,75.16,1
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z,19.9,16.05,1,35.95,1
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z,149.99,19.77,1,169.76,2


In [0]:
#Validate the Gold sales table
gold_sales.select(
    F.sum(F.when(F.col("total_items").isNull(), 1).otherwise(0))
     .alias("orders_without_items"),
    F.sum(F.when(F.col("total_payment_value").isNull(), 1).otherwise(0))
     .alias("orders_without_payments"),
    F.sum(F.when(F.col("total_product_value").isNull(), 1).otherwise(0))
     .alias("orders_without_product_value"),
    F.sum(F.when(F.col("total_freight_value").isNull(), 1).otherwise(0))
     .alias("orders_without_freight_value")
).show()

+--------------------+-----------------------+----------------------------+----------------------------+
|orders_without_items|orders_without_payments|orders_without_product_value|orders_without_freight_value|
+--------------------+-----------------------+----------------------------+----------------------------+
|                 775|                      3|                         775|                         775|
+--------------------+-----------------------+----------------------------+----------------------------+



In [0]:
#Investigate the 775 orders
gold_sales.filter(
    F.col("total_items").isNull()
).groupBy("order_status").count().orderBy(
    F.desc("count")
).show()

+------------+-----+
|order_status|count|
+------------+-----+
| unavailable|  603|
|    canceled|  164|
|     created|    5|
|    invoiced|    2|
|     shipped|    1|
+------------+-----+



Most are unavailable or canceled (767 of 775)

In [0]:
#Inspect the 8 orders with other statuses
display(
    gold_sales.filter(
        F.col("total_items").isNull() &
        ~F.col("order_status").isin("unavailable", "canceled")
    ).select(
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "total_payment_value"
    )
)

order_id,order_status,order_purchase_timestamp,total_payment_value
b5359909123fa03c50bdb0cfed07f098,created,2017-12-05T01:07:52.000Z,160.94
dba5062fbda3af4fb6c33b1e040ca38f,created,2018-02-09T17:21:04.000Z,137.6
7a4df5d8cff4090e541401a20a22bb80,created,2017-11-25T11:10:33.000Z,37.84
a68ce1686d536ca72bd2dadc4b8671e5,shipped,2016-10-05T01:47:40.000Z,77.73
35de4050331c6c644cddc86f4f2d0d64,created,2017-12-05T01:07:58.000Z,105.87
2ce9683175cdab7d1c95bcbb3e36f478,invoiced,2016-10-05T21:03:33.000Z,73.04
90ab3e7d52544ec7bc3363c82689965f,created,2017-11-06T13:12:34.000Z,245.85
e04f1da1f48bf2bbffcf57b9824f76e1,invoiced,2016-10-05T13:22:20.000Z,76.19


In [0]:
#Check Bronze order items
missing_item_orders = gold_sales.filter(
    F.col("total_items").isNull()
).select("order_id")

bronze_item_matches = order_items_silver.join(
    missing_item_orders,
    on="order_id",
    how="inner"
)

print("Matching Bronze order-item rows:", bronze_item_matches.count())
display(bronze_item_matches)

Matching Bronze order-item rows: 0


order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


The 8 orders have no matching item records in Bronze either, so the missing item data wasn't caused by our Silver processing.
For gold_sales, we'll keep these orders with null item totals rather than inventing values or excluding the orders. This preserves the original order information.

In [0]:
#Check the 3 orders missing payment records
display(
    gold_sales.filter(
        F.col("total_payment_value").isNull()
    ).select(
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "total_items",
        "total_product_value",
        "total_freight_value"
    )
)

order_id,order_status,order_purchase_timestamp,total_items,total_product_value,total_freight_value
bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15T12:16:38.000Z,3,134.97,8.49
744bade1fcf9ff3f31d860ace076d422,delivered,2018-04-22T11:34:42.000Z,1,45.9,12.79
1a57108394169c0b47d8f876acc9ba2d,delivered,2018-05-15T16:25:14.000Z,2,83.38,46.56


All 3 orders missing payment records are marked delivered and have item records in Gold. This means we should preserve their item totals and leave their payment totals null rather than assume they were paid.

In [0]:
#Finalize and save gold_sales
gold_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_path}/gold_sales")

In [0]:
#Reading gold_sales
gold_sales_check = spark.read \
    .format("delta") \
    .load(f"{gold_path}/gold_sales")

print("Saved Gold sales count:", gold_sales_check.count())
gold_sales_check.printSchema()
display(gold_sales_check.limit(10))

Saved Gold sales count: 99441
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- total_product_value: double (nullable = true)
 |-- total_freight_value: double (nullable = true)
 |-- total_items: long (nullable = true)
 |-- total_payment_value: double (nullable = true)
 |-- payment_count: long (nullable = true)



order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,total_product_value,total_freight_value,total_items,total_payment_value,payment_count
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,29.99,8.72,1,38.71,3
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,118.7,22.76,1,141.46,1
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,159.9,19.22,1,179.12,1
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,45.0,27.2,1,72.2,1
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z,19.9,8.72,1,28.62,1
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z,147.9,27.36,1,175.26,1
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z,49.9,16.05,1,65.95,1
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z,59.99,15.17,1,75.16,1
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z,19.9,16.05,1,35.95,1
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z,149.99,19.77,1,169.76,2


In [0]:
#Create gold_delivery
gold_delivery = orders_silver.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
)

gold_delivery = (
    gold_delivery
    .withColumn(
        "actual_delivery_days",
        F.when(
            F.col("order_delivered_customer_date").isNotNull(),
            F.datediff(
                F.to_date("order_delivered_customer_date"),
                F.to_date("order_purchase_timestamp")
            )
        )
    )
    .withColumn(
        "estimated_delivery_days",
        F.datediff(
            F.to_date("order_estimated_delivery_date"),
            F.to_date("order_purchase_timestamp")
        )
    )
    .withColumn(
        "delivery_delay_days",
        F.when(
            F.col("order_delivered_customer_date").isNotNull(),
            F.datediff(
                F.to_date("order_delivered_customer_date"),
                F.to_date("order_estimated_delivery_date")
            )
        )
    )
)

In [0]:
#Create a delivery status
gold_delivery = gold_delivery.withColumn(
    "delivery_status",
    F.when(
        F.col("order_status") == "delivered",
        F.when(
            F.col("order_delivered_customer_date").isNull(),
            "not_delivered"
        )
        .when(
            F.col("delivery_delay_days") > 0,
            "delayed"
        )
        .otherwise("on_time")
    )
    .otherwise("not_delivered")
)

In [0]:
print("Gold delivery count:", gold_delivery.count())

gold_delivery.printSchema()

display(gold_delivery.limit(10))

Gold delivery count: 99441
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- actual_delivery_days: integer (nullable = true)
 |-- estimated_delivery_days: integer (nullable = true)
 |-- delivery_delay_days: integer (nullable = true)
 |-- delivery_status: string (nullable = false)



order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,actual_delivery_days,estimated_delivery_days,delivery_delay_days,delivery_status
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,8,16,-8,on_time
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,14,20,-6,on_time
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,9,27,-18,on_time
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,14,27,-13,on_time
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z,3,13,-10,on_time
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z,17,23,-6,on_time
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z,null,28,null,not_delivered
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z,10,22,-12,on_time
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z,10,42,-32,on_time
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z,18,25,-7,on_time


In [0]:
#Check delivery status distribution
gold_delivery.groupBy(
    "order_status",
    "delivery_status"
).count().orderBy(
    "order_status",
    "delivery_status"
).show()

+------------+---------------+-----+
|order_status|delivery_status|count|
+------------+---------------+-----+
|    approved|  not_delivered|    2|
|    canceled|  not_delivered|  625|
|     created|  not_delivered|    5|
|   delivered|        delayed| 6534|
|   delivered|  not_delivered|    8|
|   delivered|        on_time|89936|
|    invoiced|  not_delivered|  314|
|  processing|  not_delivered|  301|
|     shipped|  not_delivered| 1107|
| unavailable|  not_delivered|  609|
+------------+---------------+-----+



In [0]:
#Check for impossible negative delivery durations
gold_delivery.select(
    F.sum(
        F.when(F.col("actual_delivery_days") < 0, 1).otherwise(0)
    ).alias("negative_actual_delivery_days"),
    F.sum(
        F.when(F.col("estimated_delivery_days") < 0, 1).otherwise(0)
    ).alias("negative_estimated_delivery_days")
).show()

+-----------------------------+--------------------------------+
|negative_actual_delivery_days|negative_estimated_delivery_days|
+-----------------------------+--------------------------------+
|                            0|                               0|
+-----------------------------+--------------------------------+



In [0]:
#Save gold_delivery
gold_delivery.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_path}/gold_delivery")

In [0]:
#Reading gold_delivery
gold_delivery_check = spark.read.format("delta").load(f"{gold_path}/gold_delivery")

print("Saved Gold delivery count:", gold_delivery_check.count())

gold_delivery_check.printSchema()

display(gold_delivery_check.limit(10))

Saved Gold delivery count: 99441
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- actual_delivery_days: integer (nullable = true)
 |-- estimated_delivery_days: integer (nullable = true)
 |-- delivery_delay_days: integer (nullable = true)
 |-- delivery_status: string (nullable = true)



order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,actual_delivery_days,estimated_delivery_days,delivery_delay_days,delivery_status
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,8,16,-8,on_time
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,14,20,-6,on_time
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,9,27,-18,on_time
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,14,27,-13,on_time
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z,3,13,-10,on_time
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z,17,23,-6,on_time
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z,null,28,null,not_delivered
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z,10,22,-12,on_time
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z,10,42,-32,on_time
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z,18,25,-7,on_time


In [0]:
#Create gold_reviews
gold_reviews = order_reviews_silver.select(
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
)

In [0]:
#add the review category
gold_reviews = gold_reviews.withColumn(
    "review_sentiment_category",
    F.when(F.col("review_score").isin(1, 2), "negative")
     .when(F.col("review_score") == 3, "neutral")
     .when(F.col("review_score").isin(4, 5), "positive")
)

In [0]:
#inspect gold_reviews
print("Gold reviews count:", gold_reviews.count())

gold_reviews.printSchema()

display(gold_reviews.limit(10))

Gold reviews count: 98410
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)
 |-- review_sentiment_category: string (nullable = true)



review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,review_sentiment_category
0001239bc1de2e33cb583967c2ca4c67,fc046d7776171871436844218f817d7d,5,null,null,2018-03-20 00:00:00,2018-03-20 18:36:04,positive
0001cc6860aeaf5b9017fe4131a52e62,d4665434b01caa9dc3e3e78b3eb3593e,5,null,null,2018-06-22 00:00:00,2018-06-26 13:51:29,positive
00020c7512a52e92212f12d3e37513c0,e28abf2eb2f1fbcbdc2dd0cd9a561671,5,Entrega rápida!,A entrega foi super rápida e o pendente é lindo! Igual a foto mesmo!,2018-04-25 00:00:00,2018-04-26 14:55:36,positive
00032b0141443497c898b3093690af51,04fb47576993a3cb0c12d4b25eab6e4e,5,null,null,2017-05-30 00:00:00,2017-06-01 23:28:55,positive
00034d88989f9a4c393bdcaec301537f,5f358d797a49fe2f24352f73426215f6,5,null,null,2017-08-12 00:00:00,2017-08-13 19:56:53,positive
000359bceee1b6fe876b30d35b5f9ef2,97868e4c884bc85f4501ae0a66181cc9,5,null,null,2018-02-06 00:00:00,2018-02-06 20:54:37,positive
00041b1f51e77a663df6db3f539d694f,998b1beee068744ac833dd8b31944623,1,null,null,2018-05-24 00:00:00,2018-05-24 12:45:38,negative
00046a69550325aea5fb89f65c7387f2,9fbda7367628952bc36c3512c46d887b,5,SUPER RECOMENDO,"GOSTEI DA CAPINHA DE CELULAR, VEIO COMO EU ESPERAVA.",2018-07-01 00:00:00,2018-07-02 12:32:24,positive
0005534973388c830bb858cfba83b17b,a589caa6892ceacc6bbf2f8cc30a8ad4,5,null,otimo produto. prazo cumprido. sabor tambem muito bom!,2017-09-01 00:00:00,2017-09-02 13:04:08,positive
00055e36e9608fe969231e551983a69c,f5fea26ab547eec920e6f8ecdc5c37e4,5,null,O produto foi entregue muito antes do esperado. Ótima agilidade.,2017-11-25 00:00:00,2017-11-27 11:37:22,positive


In [0]:
#validate the score distribution
gold_reviews.groupBy(
    "review_score",
    "review_sentiment_category"
).count().orderBy(
    "review_score"
).show()

+------------+-------------------------+-----+
|review_score|review_sentiment_category|count|
+------------+-------------------------+-----+
|           1|                 negative|11282|
|           2|                 negative| 3114|
|           3|                  neutral| 8097|
|           4|                 positive|19007|
|           5|                 positive|56910|
+------------+-------------------------+-----+



In [0]:
#Validate the review categories
gold_reviews.select(
    F.sum(F.when(F.col("review_score").isNull(), 1).otherwise(0))
     .alias("null_review_scores"),
    F.sum(F.when(F.col("review_sentiment_category").isNull(), 1).otherwise(0))
     .alias("null_sentiment_categories")
).show()

+------------------+-------------------------+
|null_review_scores|null_sentiment_categories|
+------------------+-------------------------+
|                 0|                        0|
+------------------+-------------------------+



In [0]:
#check that review_id is still unique
print(
    "Duplicate review IDs:",
    gold_reviews.groupBy("review_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

Duplicate review IDs: 0


In [0]:
#Save gold_reviews
gold_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_path}/gold_reviews")

In [0]:
#Reading gold_reviews
gold_reviews_check = spark.read.format("delta").load(f"{gold_path}/gold_reviews")

print("Saved Gold reviews count:", gold_reviews_check.count())
display(gold_reviews_check.limit(10))

Saved Gold reviews count: 98410


review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,review_sentiment_category
0001239bc1de2e33cb583967c2ca4c67,fc046d7776171871436844218f817d7d,5,null,null,2018-03-20 00:00:00,2018-03-20 18:36:04,positive
0001cc6860aeaf5b9017fe4131a52e62,d4665434b01caa9dc3e3e78b3eb3593e,5,null,null,2018-06-22 00:00:00,2018-06-26 13:51:29,positive
00020c7512a52e92212f12d3e37513c0,e28abf2eb2f1fbcbdc2dd0cd9a561671,5,Entrega rápida!,A entrega foi super rápida e o pendente é lindo! Igual a foto mesmo!,2018-04-25 00:00:00,2018-04-26 14:55:36,positive
00032b0141443497c898b3093690af51,04fb47576993a3cb0c12d4b25eab6e4e,5,null,null,2017-05-30 00:00:00,2017-06-01 23:28:55,positive
00034d88989f9a4c393bdcaec301537f,5f358d797a49fe2f24352f73426215f6,5,null,null,2017-08-12 00:00:00,2017-08-13 19:56:53,positive
000359bceee1b6fe876b30d35b5f9ef2,97868e4c884bc85f4501ae0a66181cc9,5,null,null,2018-02-06 00:00:00,2018-02-06 20:54:37,positive
00041b1f51e77a663df6db3f539d694f,998b1beee068744ac833dd8b31944623,1,null,null,2018-05-24 00:00:00,2018-05-24 12:45:38,negative
00046a69550325aea5fb89f65c7387f2,9fbda7367628952bc36c3512c46d887b,5,SUPER RECOMENDO,"GOSTEI DA CAPINHA DE CELULAR, VEIO COMO EU ESPERAVA.",2018-07-01 00:00:00,2018-07-02 12:32:24,positive
0005534973388c830bb858cfba83b17b,a589caa6892ceacc6bbf2f8cc30a8ad4,5,null,otimo produto. prazo cumprido. sabor tambem muito bom!,2017-09-01 00:00:00,2017-09-02 13:04:08,positive
00055e36e9608fe969231e551983a69c,f5fea26ab547eec920e6f8ecdc5c37e4,5,null,O produto foi entregue muito antes do esperado. Ótima agilidade.,2017-11-25 00:00:00,2017-11-27 11:37:22,positive


gold_seller_performance (One row per seller)

In [0]:
#Join the Silver tables (orders_silver and sellers_silver)
seller_orders = (
    order_items_silver
    .join(
        orders_silver.select("order_id", "order_status"),
        on="order_id",
        how="left"
    )
    .join(
        sellers_silver.select("seller_id", "seller_state"),
        on="seller_id",
        how="left"
    )
)

print("Seller-order item rows:", seller_orders.count())
display(seller_orders.limit(10))

Seller-order item rows: 112650


seller_id,order_id,order_item_id,product_id,shipping_limit_date,price,freight_value,order_status,seller_state
48436dade18ac8b2bce089ec2a041202,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,2017-09-19T09:45:35.000Z,58.9,13.29,delivered,SP
dd7ddc04e1b6c2c614352b383efe2d36,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,2017-05-03T11:05:13.000Z,239.9,19.93,delivered,SP
5b51032eddd242adc84c38acab88f23d,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,2018-01-18T14:48:30.000Z,199.0,17.87,delivered,MG
9d7a1d34a5052409006425275ba1c2b4,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,2018-08-15T10:10:18.000Z,12.99,12.79,delivered,SP
df560393f3a51e74553ab94004ba5c87,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,2017-02-13T13:57:51.000Z,199.9,18.14,delivered,PR
6426d21aca402a131fc0a5d0960a3c90,00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,2017-05-23T03:55:27.000Z,21.9,12.69,delivered,SP
7040e82f899a04d1b434b795a43b4617,00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,2017-12-14T12:10:31.000Z,19.9,11.85,delivered,SP
5996cddab893a4652a15592fb58ab8db,000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,2018-07-10T12:30:45.000Z,810.0,70.75,delivered,SP
a416b6a846a11724393025641d4edd5e,0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,2018-03-26T18:31:29.000Z,145.95,11.65,delivered,SP
ba143b05f0110f0dc71ad71b4466ce92,0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,2018-07-06T14:10:56.000Z,53.99,11.4,delivered,SP


In [0]:
#Aggregate by seller
gold_seller_performance = seller_orders.groupBy(
    "seller_id",
    "seller_state"
).agg(
    F.countDistinct("order_id").alias("total_orders"),
    F.count("order_item_id").alias("total_items_sold"),
    F.sum("price").alias("total_product_sales"),
    F.sum("freight_value").alias("total_freight_value"),
    F.countDistinct(
        F.when(F.col("order_status") == "delivered", F.col("order_id"))
    ).alias("delivered_orders"),
    F.countDistinct(
        F.when(F.col("order_status") == "canceled", F.col("order_id"))
    ).alias("canceled_orders")
)

In [0]:
#inspect the result
print("Gold seller performance count:", gold_seller_performance.count())

gold_seller_performance.printSchema()

display(
    gold_seller_performance
    .orderBy(F.desc("total_product_sales"))
    .limit(10)
)

Gold seller performance count: 3095
root
 |-- seller_id: string (nullable = true)
 |-- seller_state: string (nullable = true)
 |-- total_orders: long (nullable = false)
 |-- total_items_sold: long (nullable = false)
 |-- total_product_sales: double (nullable = true)
 |-- total_freight_value: double (nullable = true)
 |-- delivered_orders: long (nullable = false)
 |-- canceled_orders: long (nullable = false)



seller_id,seller_state,total_orders,total_items_sold,total_product_sales,total_freight_value,delivered_orders,canceled_orders
4869f7a5dfa277a7dca6462dcf3b52b2,SP,1132,1156,229472.6299999981,20168.069999999963,1124,1
53243585a1d6dc2643021fd1853d8905,BA,358,410,222776.04999999952,13080.630000000008,348,0
4a3ca9315b744ce9f8e9374361493884,SP,1806,1987,200472.9199999949,35067.04000000001,1772,2
fa1c13f2614d7b5c4749cbc52fecda94,SP,585,586,194042.02999999846,10042.700000000004,578,1
7c67e1448b00f6e969d365cea6b010ab,SP,982,1364,187923.8899999995,51612.54999999984,973,0
7e93a43ef30c4f03f38b393420bc753a,SP,336,340,176431.86999999982,6322.180000000003,319,4
da8622b14eb17ae2831f4ac5b9dab84a,SP,1314,1551,160236.56999999538,24955.75000000002,1311,0
7a67c85e85bb2ce8582c35f2203ad736,SP,1160,1171,141745.53000000177,20902.850000000002,1145,1
1025f0e2d44d7041d6cf58b6550e0bfa,SP,915,1428,138968.5499999995,33892.14000000004,910,0
955fee9216a65b617aa5c0531780ce60,SP,1287,1499,135171.69999999914,25430.980000000054,1261,1


In [0]:
#Check for null seller IDs
gold_seller_performance.select(
    F.sum(
        F.when(F.col("seller_id").isNull(), 1).otherwise(0)
    ).alias("null_seller_ids"),
    F.sum(
        F.when(F.col("seller_state").isNull(), 1).otherwise(0)
    ).alias("null_seller_states")
).show()

+---------------+------------------+
|null_seller_ids|null_seller_states|
+---------------+------------------+
|              0|                 0|
+---------------+------------------+



In [0]:
#Check metric consistency
gold_seller_performance.select(
    F.sum(
        F.when(
            (F.col("delivered_orders") + F.col("canceled_orders"))
            > F.col("total_orders"),
            1
        ).otherwise(0)
    ).alias("invalid_order_counts")
).show()

+--------------------+
|invalid_order_counts|
+--------------------+
|                   0|
+--------------------+



In [0]:
#Save gold_seller_performance
gold_seller_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_path}/gold_seller_performance")

In [0]:
#Reading gold_seller_performance
gold_seller_check = spark.read.format("delta").load(f"{gold_path}/gold_seller_performance")

print("Saved Gold seller performance count:", gold_seller_check.count())

display(gold_seller_check.limit(10))

Saved Gold seller performance count: 3095


seller_id,seller_state,total_orders,total_items_sold,total_product_sales,total_freight_value,delivered_orders,canceled_orders
1c129092bf23f28a5930387c980c0dfc,SP,201,220,9553.619999999968,3035.6700000000037,200,0
001cca7ae9ae17fb1caed9dfb1094831,ES,200,239,25080.030000000017,8854.139999999996,195,0
0adac9fbd9a2b63cccaac4f8756c1ca8,SP,110,142,15960.949999999972,2499.2699999999986,106,3
b561927807645834b59ef0d16ba55a24,SP,85,95,4501.990000000002,1305.0400000000006,85,0
e5a38146df062edaf55c38afa99e42dc,SP,121,125,15360.070000000003,1894.520000000001,120,0
f615fe7efbef0f4f08fd3086bc7a3e60,SP,23,24,2359.270000000001,389.68999999999994,22,1
7d13fca15225358621be4086e1eb0964,SP,565,578,113628.96999999993,8632.04999999999,558,0
cd06602b43d8800bd0afad514919d35c,SP,44,53,2588.07,747.2600000000002,43,1
3dbbce181edfaa4a597ec858c382d2ec,PR,17,20,848.2999999999998,440.27,17,0
85cc55e048b6bcc90c75afe7f0e72f72,RJ,9,9,308.28,193.47999999999996,9,0


gold_product_performance (products_silver, category_translation_silver and order_items_silver)

In [0]:
#Join product information with category translation
product_details = (
    products_silver
    .join(
        category_translation_silver,
        on="product_category_name",
        how="left"
    )
)

In [0]:
#join the sales information
product_sales = (
    order_items_silver
    .join(
        product_details,
        on="product_id",
        how="left"
    )
)

In [0]:
#Aggregate by product
gold_product_performance = product_sales.groupBy(
    "product_id",
    "product_category_name",
    "product_category_name_english"
).agg(
    F.count("order_item_id").alias("total_items_sold"),
    F.countDistinct("order_id").alias("total_orders"),
    F.countDistinct("seller_id").alias("total_sellers"),
    F.sum("price").alias("total_product_sales"),
    F.sum("freight_value").alias("total_freight_value"),
    F.avg("price").alias("average_product_price")
)

In [0]:
#Inspecting the aggregate
print("Gold product performance count:", gold_product_performance.count())

gold_product_performance.printSchema()

display(
    gold_product_performance
    .orderBy(F.desc("total_product_sales"))
    .limit(10)
)

Gold product performance count: 32951
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- total_items_sold: long (nullable = false)
 |-- total_orders: long (nullable = false)
 |-- total_sellers: long (nullable = false)
 |-- total_product_sales: double (nullable = true)
 |-- total_freight_value: double (nullable = true)
 |-- average_product_price: double (nullable = true)



product_id,product_category_name,product_category_name_english,total_items_sold,total_orders,total_sellers,total_product_sales,total_freight_value,average_product_price
bb50f2e236e5eea0100680137654686c,beleza_saude,health_beauty,195,187,1,63885.0,3721.1000000000017,327.61538461538464
6cdd53843498f92890544667809f1595,beleza_saude,health_beauty,156,151,1,54730.200000000106,4363.789999999998,350.8346153846161
d6160fb7873f184099d9bc95e30376af,pcs,computers,35,35,1,48899.34,1426.8400000000001,1397.1239999999998
d1c427060a0f73f6b889a5c7c61f2ac4,informatica_acessorios,computers_accessories,343,323,1,47214.50999999998,13761.519999999997,137.65163265306117
99a4788cb24856965c36a24e339b6058,cama_mesa_banho,bed_bath_table,488,467,2,43025.56000000037,8046.040000000013,88.16713114754174
3dd2a17168ec895c781a9191c1e95ad7,informatica_acessorios,computers_accessories,274,255,1,41082.60000000021,7129.620000000003,149.93649635036573
25c38557cf793876c5abdd5931f922db,bebes,baby,38,38,2,38907.32000000001,1404.6299999999997,1023.8768421052633
5f504b3a1c75b73d6151be81eb05bdc9,cool_stuff,cool_stuff,63,63,1,37733.90000000001,3991.909999999998,598.9507936507938
53b36df67ebb7c41585e8d54d6772e08,relogios_presentes,watches_gifts,323,306,3,37683.42000000013,2274.510000000001,116.66693498452052
aca2eb7d00ea1a7b8ebd4e68314663af,moveis_decoracao,furniture_decor,527,431,1,37608.900000000314,7211.859999999983,71.36413662239148


In [0]:
#Validate product metrics
gold_product_performance.select(
    F.sum(
        F.when(F.col("product_id").isNull(), 1).otherwise(0)
    ).alias("null_product_ids"),
    F.sum(
        F.when(F.col("total_items_sold") <= 0, 1).otherwise(0)
    ).alias("invalid_item_counts"),
    F.sum(
        F.when(F.col("total_product_sales") < 0, 1).otherwise(0)
    ).alias("negative_product_sales"),
    F.sum(
        F.when(F.col("total_freight_value") < 0, 1).otherwise(0)
    ).alias("negative_freight_values")
).show()

+----------------+-------------------+----------------------+-----------------------+
|null_product_ids|invalid_item_counts|negative_product_sales|negative_freight_values|
+----------------+-------------------+----------------------+-----------------------+
|               0|                  0|                     0|                      0|
+----------------+-------------------+----------------------+-----------------------+



In [0]:
#Check product IDs are unique
print(
    "Duplicate product IDs:",
    gold_product_performance
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

Duplicate product IDs: 0


In [0]:
#Save gold_product_performance Gold table
gold_product_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_path}/gold_product_performance")

In [0]:
#Reading gold_product_performance Gold table
gold_product_check = spark.read.format("delta").load(f"{gold_path}/gold_product_performance")

print(
    "Saved Gold product performance count:",
    gold_product_check.count()
)

gold_product_check.printSchema()

display(gold_product_check.limit(10))

Saved Gold product performance count: 32951
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- total_items_sold: long (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- total_sellers: long (nullable = true)
 |-- total_product_sales: double (nullable = true)
 |-- total_freight_value: double (nullable = true)
 |-- average_product_price: double (nullable = true)



product_id,product_category_name,product_category_name_english,total_items_sold,total_orders,total_sellers,total_product_sales,total_freight_value,average_product_price
10dfb99fdae9b721c39b7458564397f6,cama_mesa_banho,bed_bath_table,2,1,1,519.8,37.06,259.9
cae7248e9bdfda82584faea19dbf793f,relogios_presentes,watches_gifts,4,4,1,196.0,81.78999999999999,49.0
7f6f8bdeee8ab42b5b4873b754f22a15,papelaria,stationery,1,1,1,189.0,8.58,189.0
1d6494a42302dba967b0c3a9c655d1a5,utilidades_domesticas,housewares,4,4,1,300.0,86.66,75.0
aa5a3fa92cb1221a38267376d25cfd8a,cama_mesa_banho,bed_bath_table,1,1,1,69.9,18.29,69.9
fbb1cfc2810efabf3235eccf4530f4ae,null,null,2,2,1,89.8,31.67,44.9
09e68b02ea2a0a677c692faac3735c9d,esporte_lazer,sports_leisure,1,1,1,69.9,44.34,69.9
3b3c3503539e7f917eb57a8d4bfa69f0,cool_stuff,cool_stuff,9,9,1,3045.0,203.28,338.3333333333333
495f9290868e1bde785d3c372608ad54,utilidades_domesticas,housewares,2,2,1,209.8,70.45,104.9
516f4c728fa63b7394b4a4d99a010ff6,null,null,1,1,1,284.9,22.07,284.9


In [0]:
print("===== GOLD TABLE VALIDATION =====")

gold_tables = {
    "gold_sales": f"{gold_path}/gold_sales",
    "gold_delivery": f"{gold_path}/gold_delivery",
    "gold_reviews": f"{gold_path}/gold_reviews",
    "gold_seller_performance": f"{gold_path}/gold_seller_performance",
    "gold_product_performance": f"{gold_path}/gold_product_performance"
}

for table_name, path in gold_tables.items():
    df = spark.read.format("delta").load(path)

    print(
        f"{table_name}: "
        f"{df.count()} rows | "
        f"{len(df.columns)} columns"
    )

===== GOLD TABLE VALIDATION =====
gold_sales: 99441 rows | 13 columns
gold_delivery: 99441 rows | 12 columns
gold_reviews: 98410 rows | 8 columns
gold_seller_performance: 3095 rows | 8 columns
gold_product_performance: 32951 rows | 9 columns


In [0]:
print("\n===== GRAIN VALIDATION =====")

print(
    "Duplicate orders in gold_sales:",
    gold_sales.groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate reviews in gold_reviews:",
    gold_reviews.groupBy("review_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate sellers in gold_seller_performance:",
    gold_seller_performance.groupBy("seller_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate products in gold_product_performance:",
    gold_product_performance.groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)


===== GRAIN VALIDATION =====
Duplicate orders in gold_sales: 0
Duplicate reviews in gold_reviews: 0
Duplicate sellers in gold_seller_performance: 0
Duplicate products in gold_product_performance: 0


Dashboard-ready summary DataFrame

In [0]:
#dashboard aggregation
dashboard_kpis = gold_sales.select(
    F.countDistinct("order_id").alias("total_orders"),
    F.sum("total_product_value").alias("total_product_sales"),
    F.sum("total_freight_value").alias("total_freight_value")
).crossJoin(
    gold_delivery.filter(
        F.col("order_status") == "delivered"
    ).select(
        F.countDistinct("order_id").alias("delivered_orders")
    )
).crossJoin(
    gold_reviews.select(
        F.avg("review_score").alias("average_review_score")
    )
)

display(dashboard_kpis)

total_orders,total_product_sales,total_freight_value,delivered_orders,average_review_score
99441,1.3591643700001858E7,2251909.539999922,96478,4.088801951021238


In [0]:
#Orders by Status
orders_by_status = (
    gold_sales
    .groupBy("order_status")
    .agg(
        F.countDistinct("order_id").alias("order_count")
    )
    .orderBy(F.desc("order_count"))
)

display(orders_by_status)

order_status,order_count
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


In [0]:
#Monthly Sales Trend
monthly_sales = (
    gold_sales
    .withColumn(
        "sales_month",
        F.date_trunc("month", "order_purchase_timestamp")
    )
    .groupBy("sales_month")
    .agg(
        F.sum("total_product_value").alias("total_product_sales"),
        F.countDistinct("order_id").alias("order_count")
    )
    .orderBy("sales_month")
)

display(monthly_sales)

sales_month,total_product_sales,order_count
2016-09-01T00:00:00.000Z,267.36,4
2016-10-01T00:00:00.000Z,49507.66000000002,324
2016-12-01T00:00:00.000Z,10.9,1
2017-01-01T00:00:00.000Z,120312.87000000023,800
2017-02-01T00:00:00.000Z,247303.02000000008,1780
2017-03-01T00:00:00.000Z,374344.2999999976,2682
2017-04-01T00:00:00.000Z,359927.2299999981,2404
2017-05-01T00:00:00.000Z,506071.13999999536,3700
2017-06-01T00:00:00.000Z,433038.59999999695,3245
2017-07-01T00:00:00.000Z,498031.4799999952,4026


In [0]:
#Delivery Performance
delivery_performance = (
    gold_delivery
    .groupBy("delivery_status")
    .agg(
        F.countDistinct("order_id").alias("order_count")
    )
    .orderBy(F.desc("order_count"))
)

display(delivery_performance)

delivery_status,order_count
on_time,89936
delayed,6534
not_delivered,2971


In [0]:
#Review Score Distribution
review_distribution = (
    gold_reviews
    .groupBy("review_score")
    .agg(
        F.count("review_id").alias("review_count")
    )
    .orderBy("review_score")
)

display(review_distribution)

review_score,review_count
1,11282
2,3114
3,8097
4,19007
5,56910
